# Trade Data Scientist Assessment

## Modules

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from trade_data.ingestion import (load_cimt_data, load_HS_dictionary, load_country_cimt_dictionary, 
                                  load_baci_cepii_data, load_cepii_country_dictionary, load_wdi_full_ind, load_wgi_full_ind, load_itc_data, load_gravity_cepii_data, load_statcan_tables)
from trade_data.processing import cimt_processing
from trade_data.analysis import top10_prod_ab, top10_country_ab, top5_prod_by_top10_ctry, trade_indices
from trade_data import modelling as M

from trade_data.ingestion import load_yale_s338, load_usitc_hts8, load_updates_proc_s338
from trade_data import s338 as S
from trade_data.analysis import (hs6_coverage, tot_can_exp_hs6, province_gdp, xposr_trade, xposr_prov, xposr_prog,
                                 xposr_hs4, province_table, hs4_table, tophs4_prov, MAJOR_PROVINCES)

In [2]:
# prepare dictionaries
HS_dict = load_HS_dictionary()
cimt_country_dict = load_country_cimt_dictionary()

## Question 1 – Alberta’s Export Growth and Diversification Potential 

In [3]:
# load and process CIMT data

cimt = load_cimt_data()
cimt = cimt_processing(cimt)

In [4]:
cimt.head()

,HS8,Country/Pays,Province,State/État,Value/Valeur,Quantity/Quantité,Unit of Measure/Unité de Mesure,Year,Month,HS4
0,01012100,US,AB,NH,12100,1,NMB,2024,01,0101
1,01012100,US,ON,FL,5045,2,NMB,2024,01,0101
2,01012100,US,ON,IL,17117,2,NMB,2024,01,0101
3,01012100,US,ON,IN,36905,6,NMB,2024,01,0101
4,01012100,US,ON,KY,53557,5,NMB,2024,01,0101


Useful elements for analysis

1- Global evolution between 2025 and 2024 (CIMT) then nominal comparison

2- Compute UVI and Volume Indices to see if the trend is from volume or value

3- Main products and partners

4- Highlight main products for top 2/3 partners

5- Put in annex the full tables (Global evolution, UVI,Vol I., Val. I. and Top 5 Products of Top 10 countries)

In [5]:
# Alberta's exports 
tot = cimt[cimt["Year"].isin(["2024", "2025"]) & cimt["Province"].eq("AB")].groupby("Year")["Value/Valeur"].sum()

v2024 = tot.loc["2024"]
v2025 = tot.loc["2025"]

var = v2025 - v2024
var_pct = var / v2024 * 100

print(f"2024 : {v2024:,.0f} CAD")
print(f"2025 : {v2025:,.0f} CAD")
print(f"Variation : {var:,.0f} CAD ({var_pct:.2f} %)")

2024 : 182,634,257,196 CAD
2025 : 177,964,624,065 CAD
Variation : -4,669,633,131 CAD (-2.56 %)


### • Using appropriate data sources identify Alberta’s top ten export destination countries in 2025, ranked by total domestic export value. 

In [6]:
#Top ten products exports in billion dollars CAD for Alberta in 2025


topten_prod = top10_prod_ab(cimt,2025) 
topten_prod
topten_prod.merge(HS_dict.set_index('ProductCode'), left_on='HS4', right_on='ProductCode', how='left')[['HS4','Product Description','Value (Billion CAD)', 'Share (%)']]

,HS4,Product Description,Value (Billion CAD),Share (%)
0,2709,Petroleum oils and oils obtained from bitumino...,119.84,67.34
1,2711,Petroleum gases and other gaseous hydrocarbons,11.76,6.61
2,3901,"Polymers of ethylene, in primary forms",3.60,2.02
3,0201,Meat of bovine animals; fresh or chilled,3.21,1.80
4,1001,Wheat and meslin,3.17,1.78
5,2710,Petroleum oils and oils from bituminous minera...,2.98,1.68
6,0102,Bovine animals; live,1.57,0.88
7,1205,Rape or colza seeds; whether or not broken,1.54,0.87
8,1514,"Rape, colza or mustard oil and their fractions...",1.30,0.73
9,4703,"Chemical wood pulp, soda or sulphate, other th...",1.24,0.69


In [7]:
trade_indices(cimt, province='AB',year1=2024, year2=2025)#.round(2)

{'UV_Indice_y2_y1': np.float64(96.70051441289739),
 'Value_Indice_y2_y1': np.float64(97.35879379787681),
 'Volume_Indice_y2_y1': np.float64(100.68074031350926),
 'Coverage_rate_y1': np.float64(96.40341874364199),
 'Coverage_rate_y2': np.float64(96.31993509866997)}

### • Develop a machine-learning model using 2024 bilateral HS4 merchandise trade flows among the United States, Canada, Mexico, China, South Korea, Japan, and the United Kingdom, together with relevant macroeconomic, social, developmental, geographic, and other measurable factors that may influence bilateral trade. 

Mmodel trained on 2024 HS4 flows among the seven panel countries (Canada included as an exporter and as a destination, zero flows included). 

Alberta is then predicted as a separate exporter: its own GDP, population, border (U.S only) and HS4 supply, with Canada values as proxies for distance, language, governance, trade openness, agreements and tariffs.

In [8]:
# Raw data (ingestion only)
cepii = load_baci_cepii_data()
cepii_country = load_cepii_country_dictionary()
wdi_data = load_wdi_full_ind("2024:2025", countries="all")
wgi_data = load_wgi_full_ind("2024:2025", countries="all")
datadict = load_statcan_tables()
grav_init = load_gravity_cepii_data(ind=["distw_harmonic", "contig", "comlang_off", "comlang_ethno",
                                         "comcol", "col45", "fta_wto"], year=2021)
itc = load_itc_data()
rate = {2024: 1.3698, 2025: 1.3978} 

In [9]:
# Reference tables for the model: world HS4 flows, Alberta exports, macro, gravity, tariffs
inputs = M.prepare_inputs(cimt, cepii, cepii_country, wdi_data, wgi_data, datadict, grav_init, itc, rate)
del cepii    # free memory (~4.5 M rows kept in inputs["flows"])

print(inputs["flows"].shape, inputs["flows"].duplicated(["o", "d", "hs4"]).sum(), len(inputs["hs4"]))
print(inputs["macro"].duplicated(["country", "year"]).sum())
inputs["macro"][inputs["macro"]["country"].isin(["AB", "CAN"])]

(4495446, 4) 0 1228
0


,country,year,gdp_usd,gdp_per_cap_usd,pop,trade_op_pct_gdp,rule_of_law_est,regulatory_quality_est
70,CAN,2024,2.270076e+12,55015.706692,41262329,65.121499,1.482008,1.287155
71,CAN,2025,2.319900e+12,55697.663966,41651653,63.510505,1.476916,1.440638
522,AB,2024,3.459899e+11,70584.331972,4901795,65.121499,1.482008,1.287155
523,AB,2025,3.451623e+11,68694.059761,5024631,63.510505,1.476916,1.440638


In [10]:
# Training table (2024, panel) and prediction table (Alberta 2025, all destinations)
train = M.training_set(inputs)
ab_world = M.alberta_set(inputs)
M.check_tables(train, ab_world)

Training rows: 51,576 | share of zero flows: 18.0%
Missing values ('ave' could be > 0):
gdp_usd_o         0.000
gdp_usd_d         0.000
distw_harmonic    0.000
contig            0.000
ave               0.144
Alberta -> USA: GDP 3.452e+11 USD, contig 1, fta 1
Alberta destinations: 194


XGBoost with a Poisson objective (count:poisson): the machine-learning counterpart of the PPML gravity model, which keeps zero flows and treats large and small flows consistently.

In [11]:
# Print Validation test results when we train the model leaving one country out and checking how it predicts

cv = M.leave_one_exporter_out(train)
display(cv.round(3))
cv.median().round(3)

,WAPE_flow,Spearman_flow,AUC_presence,WAPE_destination,Allocated_flow,Allocated_destination
held_out_exporter,,,,,,
USA,0.660,0.905,0.903,0.545,0.649,0.895
CAN,0.682,0.781,0.835,0.544,0.631,0.875
MEX,0.495,0.775,0.852,0.410,0.769,0.905
CHN,0.785,0.855,0.870,0.519,0.547,0.905
KOR,0.704,0.838,0.862,0.368,0.630,0.837
JPN,0.667,0.847,0.853,0.317,0.651,0.947
GBR,0.735,0.787,0.824,0.216,0.676,0.937


WAPE_flow                0.682
Spearman_flow            0.838
AUC_presence             0.853
WAPE_destination         0.410
Allocated_flow           0.649
Allocated_destination    0.905
dtype: float64

When dealing with an unknown exporter, the model ranks trade flows accurately (Spearman 0.84) and correctly identifies who trades with whom (AUC 0.85). 

However,  flow values are imprecise (WAPE 0.68; 65% of value correctly allocated across destination-product pairs, 90% across destinations). Predictions are therefore rescaled to Alberta's actual supply: the model is used for the geographic distribution, not the level.

In [12]:
# Train the model and check the features importance
model = M.fit_xgb(train)
M.feature_importance(model)

log_supply                  31.9
log_demand                  19.2
contig                      15.6
log_distw_harmonic           3.4
log_gdp_usd_d                3.2
log_pop_o                    3.2
log_pop_d                    2.5
log_gdp_usd_o                2.4
regulatory_quality_est_o     1.9
fta                          1.9
gdp_per_cap_usd_d            1.8
hs2                          1.7
trade_op_pct_gdp_o           1.4
comlang_ethno                1.3
gdp_per_cap_usd_o            1.3
regulatory_quality_est_d     1.3
rule_of_law_est_d            1.3
comlang_off                  1.3
rule_of_law_est_o            1.2
trade_op_pct_gdp_d           1.1
ave                          0.8
col45                        0.5
Name: gain share (%), dtype: float64

Supply, demand and sharing borders with the destination are the top features.

In [13]:
# Compute the potential and add it to ab_world

ab_world = M.add_potential(ab_world, model, inputs)
ab_world[["d", "hs4", "pred", "pred_constrained", "Observed 2025 (M CAD)", "Potential 2025 (M CAD)"]].head()

,d,hs4,pred,pred_constrained,Observed 2025 (M CAD),Potential 2025 (M CAD)
0,ABW,0101,82.598137,41.929005,0.0,0.058608
1,ABW,0102,490.011475,368.620352,0.0,0.515258
2,ABW,0103,83.739754,41.499380,0.0,0.058008
3,ABW,0104,13.738367,0.028080,0.0,0.000039
4,ABW,0105,13.832550,0.215293,0.0,0.000301


In [14]:
# Validations on model's precision (Alberta 2025 vs Naive) and ability to spot new markets

scores, new_markets = M.validate_alberta(ab_world, inputs)
display(scores.round(3))
new_markets.round(3)

,WAPE_flow,Spearman_flow,AUC_presence,WAPE_destination,Allocated_flow,Allocated_destination
XGBoost (unconstrained),0.476,0.589,0.876,0.315,0.805,0.949
XGBoost (supply-constrained),0.271,0.530,0.832,0.193,0.880,0.969
Naive: 2025 is equal to 2024,0.191,0.805,0.884,0.105,0.926,0.975


,AUC,Openings in top 10%,Value in top 10%,Lift top 10%
Prediction made by XGBoost,0.816,0.413,0.897,4.125
Canada already exports the HS4 to d (2024),0.840,0.562,0.620,5.617


I -
Average precision (Naive beats XGBoosts)

II -
Among destination–product pairs with no Alberta exports in 2024, the 10% ranked highest by the model contain 90% of the value of trade that opened in 2025, but only 41% of the openings. 
The model is then best to find where the largest opportunities lie, rather than to which small flows will start.

In [15]:
# Top 10 countries : 2025 potential and observed exports, by HS4 products
hs_names = HS_dict.set_index("ProductCode")["Product Description"]
ctry_names = cepii_country.drop_duplicates("country_iso3").set_index("country_iso3")["country_name"]
pd.set_option("display.float_format", "{:,.1f}".format)

# Print top 5 for top 10's (All countries, not just the panel)
ab_panel = ab_world[ab_world["d"].isin(M.PARTNERS)]
#dest_panel, top5_panel = M.top_tables(ab_panel, "Potential 2025 (M CAD)", hs_names, ctry_names)
dest_world, top5_world = M.top_tables(ab_world, "Potential 2025 (M CAD)", hs_names, ctry_names)
#display(dest_panel)

# Print top 10 predicted countries in 2025
display(dest_world)

# Prints top 5 products for each country of the top 10
top5_world

,Country,Potential 2025 (M CAD),Observed 2025 (M CAD),In training panel
d,,,,
USA,USA,"124,152.2","151,374.3",True
CHN,China,"9,414.1","10,192.6",True
ESP,Spain,"3,302.2",179.4,False
GBR,United Kingdom,"3,290.5",181.9,True
NLD,Netherlands,"2,040.8",541.0,False
FRA,France,"1,882.8",242.1,False
JPN,Japan,"1,765.2","2,387.6",True
BEL,Belgium,"1,658.6",176.0,False
KOR,Rep. of Korea,"1,643.4","1,393.2",True


,rank,Country,hs4,Product,Potential 2025 (M CAD),Observed 2025 (M CAD)
226209,1,USA,2709,Petroleum oils and oils obtained from bitumino...,"99,421.3","110,821.6"
226211,1,USA,2711,Petroleum gases and other gaseous hydrocarbons,"4,431.6","11,573.4"
226396,1,USA,3901,"Polymers of ethylene, in primary forms","2,181.4","2,633.2"
225958,1,USA,0201,Meat of bovine animals; fresh or chilled,"1,477.0","2,845.3"
225953,1,USA,0102,Bovine animals; live,"1,242.7","1,572.5"
40781,2,China,2709,Petroleum oils and oils obtained from bitumino...,"7,211.3","6,319.5"
40783,2,China,2711,Petroleum gases and other gaseous hydrocarbons,407.8,16.5
40631,2,China,1205,Rape or colza seeds; whether or not broken,375.3,431.0
41060,2,China,4703,"Chemical wood pulp, soda or sulphate, other th...",216.2,437.3
40968,2,China,3901,"Polymers of ethylene, in primary forms",192.3,588.6


Observed values exclude Canadian special codes (HS 98–99), hence USD 151.4 B vs 152.0 B in the CIMT total

In [16]:
# Check our results  without fuel HS27 to see its impact on our predictions
#  
no_fuel, totals = M.without_fuels(ab_world)
print(totals)
dest_nf, top5_nf = M.top_tables(no_fuel, "Unrealized 2025 (M CAD)", hs_names, ctry_names,
                                other_cols=("Observed 2025 (M CAD)", "Potential 2025 (M CAD)"))

# Top 10 countries / energy (HS27) excluded
display(dest_nf) # Countries only
top5_nf # countries and top 5 HS4

{'Observed 2025 (B CAD)': np.float64(40.778443935999995), 'Unrealized potential (B CAD)': np.float64(22.026609688169316)}


,Country,Unrealized 2025 (M CAD),Observed 2025 (M CAD),Potential 2025 (M CAD),In training panel
d,,,,,
USA,USA,"2,110.1","26,204.9","18,178.9",True
DEU,Germany,552.6,107.3,626.9,False
NLD,Netherlands,422.8,540.7,550.9,False
GBR,United Kingdom,373.3,181.8,461.2,True
ITA,Italy,368.3,310.2,506.0,False
CHN,China,355.1,"3,616.5","1,694.7",True
BEL,Belgium,341.5,175.8,429.0,False
FRA,France,341.1,242.1,431.3,False
MEX,Mexico,335.1,949.1,705.5,True


,rank,Country,hs4,Product,Unrealized 2025 (M CAD),Observed 2025 (M CAD),Potential 2025 (M CAD)
226012,1,USA,0713,"Vegetables, leguminous; shelled, whether or no...",345.8,85.8,431.5
226271,1,USA,2905,"Acyclic alcohols and their halogenated, sulpho...",243.7,405.1,648.8
226834,1,USA,7502,Nickel; unwrought,242.5,0.0,242.5
226086,1,USA,1514,"Rape, colza or mustard oil and their fractions...",198.8,949.8,"1,148.5"
226143,1,USA,2306,Oil-cake and other solid residues; whether or ...,161.1,470.5,631.6
56494,2,Germany,0201,Meat of bovine animals; fresh or chilled,89.6,0.1,89.6
56595,2,Germany,1205,Rape or colza seeds; whether or not broken,54.9,31.7,86.7
56574,2,Germany,1001,Wheat and meslin,52.3,0.0,52.3
56932,2,Germany,3901,"Polymers of ethylene, in primary forms",49.5,0.5,50.1
57024,2,Germany,4703,"Chemical wood pulp, soda or sulphate, other th...",24.2,0.8,25.0


Comments
Effectively observed exports in 2025:
The U.S. accounted for C$152B, followed by China C$10B and Japan C$2B. Crude oil alone accounted for 67%. 

Model-estimated potential for 2025:
The U.S. is still ranked first but is below the observed level (C$124B vs C$152B):: Alberta is more reliant on the United States than trade fundamentals suggest. Spain, the Netherlands, France, Belgium, and Italy (five of the anticipated top ten) are not included in the training panel (extrapolation).

For beef, canola, wheat, and polyethylene, Europe (Germany, Netherlands, United Kingdom, Italy) has the biggest gaps after the United States (C$2B), with C$41B observed and C$22B unrealized potential, excluding mineral fuels.


## Question 2 – Canada’s Exposure to U.S. Section 338 Trade Measures 

In [17]:
# Load and prepare Section 338 tables
s338_jul = load_yale_s338()
upd = load_updates_proc_s338()
hts8_schedule = load_usitc_hts8()

s338 = S.updated_list(s338_jul, upd)
coverage = hs6_coverage(s338, hts8_schedule) # covered lines by type of coverage (full, weighted,...)

print("HTS8 lines July list:", len(s338_jul), "| latest list:", len(s338),
      "| of which import ban:", int(s338["import_ban"].sum()))
print("HS6 touched:", len(coverage), "| fully covered:", int((coverage["line_share"] == 1).sum()))

############################
s338 = s338.groupby(["status", "program"]).size()
s338.rename("HTS8 lines").to_frame()

HTS8 lines July list: 554 | latest list: 659 | of which import ban: 64
HS6 touched: 420 | fully covered: 249


HTS8 lines
status    program                   
July list alcohol                 63
          dairy                   52
          motor_vehicles         434
sep. 15   alcohol                 34
          motor_vehicles          76

In [18]:
# load and prepare hs6 exports, gdp, and compute exposure with up to date list by product / province
exports = tot_can_exp_hs6(cimt, 2025)
gdp = province_gdp(datadict, 2024)
us = xposr_trade(exports, coverage)
prov = xposr_prov(exports, us, gdp)

# Same as previous with July list
cov_july = hs6_coverage(s338_jul.assign(import_ban=False), hts8_schedule)
prov_july = xposr_prov(exports, xposr_trade(exports, cov_july), gdp)
print(f"Canada exposure /July list: {prov_july.loc['Canada', 'exposure'] / 1e9:,.1f} Billions CAD | "
      f"latest list: {prov.loc['Canada', 'exposure'] / 1e9:,.1f} Billions CAD")
pd.set_option("display.float_format", "{:,.2f}".format)
province_table(prov)

Canada exposure /July list: 33.2 Billions CAD | latest list: 37.1 Billions CAD


,"Exposure, HS6 (B CAD)",Lower bound: fully listed HS6 (B CAD),exposure_weighted,of which import ban (B CAD),Exports to U.S. (B CAD),Total exports (B CAD),GDP 2024 (B CAD),Share of Canada (%),% of total exports,% of exports to U.S.,% of GDP
Province,,,,,,,,,,,
Canada,37.09,14.80,22.93,1.54,517.28,720.77,"3,107.67",100.00,5.15,7.17,1.19
ON,18.90,8.81,12.26,0.92,190.32,265.98,"1,197.02",50.96,7.11,9.93,1.58
QC,10.47,3.32,6.12,0.54,84.34,117.64,616.77,28.23,8.90,12.41,1.70
BC,3.83,1.33,2.20,0.03,27.92,54.74,429.09,10.32,6.99,13.72,0.89
AB,1.74,0.55,0.96,0.05,152.04,177.96,473.94,4.68,0.98,1.14,0.37
MB,0.84,0.34,0.55,0.00,12.94,19.64,96.12,2.25,4.25,6.45,0.87
NS,0.54,0.12,0.30,0.00,4.58,6.61,65.34,1.45,8.14,11.74,0.82
NB,0.45,0.14,0.30,0.01,14.47,16.01,48.30,1.21,2.80,3.10,0.93
SK,0.29,0.16,0.21,0.00,23.68,43.58,112.84,0.78,0.67,1.23,0.26


In [19]:
# Province's exposure in 2025 by annexes of the proclamation
display((xposr_prog(us).loc[MAJOR_PROVINCES + ["Canada"]] / 1e9).round(2))

# Show expose by major provinces
tophs4_prov(exports, us, hs_names)

program,alcohol,alcohol/motor_vehicles,dairy,dairy/motor_vehicles,motor_vehicles
Province,,,,,
ON,1.19,0.24,0.19,0.51,16.76
QC,0.63,0.13,0.03,0.06,9.61
AB,0.18,0.00,0.00,0.00,1.55
BC,0.15,0.01,0.02,0.06,3.59
SK,0.01,0.00,0.04,0.00,0.24
MB,0.04,0.00,0.00,0.10,0.69
Canada,2.22,0.40,0.29,0.74,33.44


,Province,HS4,Product,Exposure (Millions CAD),Intensity (%),Share of province exposure (%)
0,ON,3923,Plastic articles for the conveyance or packing...,"1,506.44",67.45,7.97
1,ON,9401,"Seats (not those of heading no. 9402), whether...","1,426.46",92.96,7.55
2,ON,9403,"Furniture and parts thereof, n.e.c. in chapter 94","1,341.17",64.42,7.10
3,QC,4802,"Uncoated paper and paperboard, used for writin...",874.00,70.06,8.35
4,QC,9403,"Furniture and parts thereof, n.e.c. in chapter 94",742.01,88.79,7.09
5,QC,9405,Luminaires and light fittings; including searc...,638.23,88.33,6.10
6,AB,8414,"Air or vacuum pumps, air or other gas compress...",214.11,78.14,12.32
7,AB,8537,"Boards, panels, consoles, desks, cabinets, bas...",175.51,85.72,10.10
8,AB,9403,"Furniture and parts thereof, n.e.c. in chapter 94",166.04,90.78,9.56
9,BC,8537,"Boards, panels, consoles, desks, cabinets, bas...",913.76,90.90,23.86


In [20]:
# Alberta's exposure in 2025 by HS4 products

print(len(xposr_hs4(exports, us, "AB")), "exposed HS4 for Alberta")
ab_hs4 = hs4_table(exports, us, "AB", hs_names, n=15)
print(f"Top 15 = {ab_hs4['Share of province exposure (%)'].sum():.0f}% of Alberta's exposure")
ab_hs4

162 exposed HS4 for Alberta
Top 15 = 80% of Alberta's exposure


,Product,"Exposure, HS6 (Millions CAD)",Lower bound: fully listed HS6 (Millions CAD),exposure_weighted,Import ban (Millions CAD),"Province exports of HS4, all destinations (Millions CAD)",Exposure intensity (%),Share of province exposure (%),Min. U.S. line share
hs4,,,,,,,,,
8414,"Air or vacuum pumps, air or other gas compress...",214.11,0.00,53.53,0.00,274.00,78.14,12.32,0.25
8537,"Boards, panels, consoles, desks, cabinets, bas...",175.51,0.00,43.88,0.00,204.76,85.72,10.10,0.25
9403,"Furniture and parts thereof, n.e.c. in chapter 94",166.04,109.07,138.58,0.00,182.89,90.78,9.56,0.33
3824,Prepared binders for foundry moulds or cores; ...,159.62,0.00,23.94,0.00,173.68,91.90,9.19,0.15
8526,"Radar apparatus, radio navigational aid appara...",95.21,94.53,94.87,0.00,155.69,61.15,5.48,0.50
4411,Fibreboard of wood or other ligneous materials...,83.70,0.00,50.06,0.00,85.25,98.18,4.82,0.40
8419,"Machinery, plant (not domestic), or laboratory...",73.69,0.00,24.56,0.00,255.78,28.81,4.24,0.33
3923,Plastic articles for the conveyance or packing...,72.91,70.87,71.89,0.00,104.67,69.65,4.20,0.50
4811,"Paper, paperboard, cellulose wadding and webs ...",66.71,0.00,22.23,0.00,67.39,98.99,3.84,0.29
